# Neural Networks and Deep Learning — Seminar 10: Reinforcement learning: trading, hedging and battery dispatch
Prof. dr. Daniel Traian Pele — Bucharest University of Economic Studies (ASE); Romanian Academy, Institute for Economic Forecasting (IPE)

[Open in Colab](https://colab.research.google.com/github/danpele/NEURAL_BIZ/blob/main/notebooks/L10_seminar.ipynb)

Seminar tasks on real data from `../data/clean` (see `../data/SOURCES.md`). Write your code in the empty cells under each task; keep the seed fixed so that your numbers can be compared.

In [ ]:
# Google Colab: copy the course repository, so that the data and the saved results are found
import os, subprocess, sys
if 'google.colab' in sys.modules and not os.getcwd().startswith('/content/NEURAL_BIZ'):
    if not os.path.isdir('/content/NEURAL_BIZ'):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/danpele/NEURAL_BIZ.git', '/content/NEURAL_BIZ'], check=True)
    os.chdir('/content/NEURAL_BIZ/notebooks')

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
DATA = '../data/clean/'
SEED = 20261108
random.seed(SEED); np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
except ImportError:
    DEVICE = 'cpu'
plt.rcParams.update({'figure.facecolor': 'none', 'axes.facecolor': 'none', 'axes.grid': False,
                     'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False})
print('device:', DEVICE)

## Setup: prices and the battery model

In [ ]:
import torch, torch.nn as nn
x = pd.read_csv(DATA + 'power_da_prices_hourly_utc.csv', parse_dates=['timestamp_utc']).set_index('timestamp_utc')
x.index = x.index.tz_convert('Europe/Brussels')
END = '2026-10-31'                                  # last complete month of the course data
x = x.loc[:END + ' 23:59']
x['day'] = x.index.tz_localize(None).normalize(); x['hour'] = x.index.hour

def day_matrix(zone):
    # delivery days (CET) x 24 hours; only days with 24 reported hours
    d = x[['day', 'hour', zone]].dropna()
    n = d.groupby('day')[zone].count()
    return d[d.day.isin(n[n == 24].index)].pivot_table(index='day', columns='hour', values=zone).sort_index()

RO = day_matrix('RO')
print(RO.index.min().date(), RO.index.max().date(), len(RO), 'complete days')

In [ ]:
E_MAX, ETA = 2, 0.85                 # 2 MWh, 1 MWh of stored energy per hour, 85 % round-trip efficiency
ETA_C = ETA_D = np.sqrt(ETA)
ACTS = np.array([-1, 0, 1])          # charge, idle, discharge

def reward(p, a):
    return np.where(a > 0, ETA_D * p, np.where(a < 0, -p / ETA_C, 0.0))

def dp_days(P):
    # backward induction V_h(s) = max_a { r(p_h, a) + V_{h+1}(s - a) }, all days at once
    D = P.shape[0]
    V = np.zeros((D, 25, E_MAX + 1)); A = np.zeros((D, 24, E_MAX + 1), dtype=int)
    for h in range(23, -1, -1):
        best = np.full((D, E_MAX + 1), -np.inf); arg = np.zeros((D, E_MAX + 1), dtype=int)
        for s in range(E_MAX + 1):
            for a in ACTS:
                if 0 <= s - a <= E_MAX:
                    q = reward(P[:, h], a) + V[:, h + 1, s - a]
                    b = q > best[:, s]; best[b, s] = q[b]; arg[b, s] = a
        V[:, h] = best; A[:, h] = arg
    return V, A

**Task 1.** Compute the perfect-foresight revenue of the battery for every year in Hungary (HU) and Germany-Luxembourg (DE_LU) and compare it with Romania. Which market paid the most in 2025?

In [ ]:
# your code here

**Task 2.** Re-run the dynamic programme with a round-trip efficiency of 0.90 and 0.75. How much does the 2025 revenue in Romania change? Explain why efficiency matters more on days with a small spread.

In [ ]:
# your code here

**Task 3.** Naive benchmark: apply yesterday's optimal plan to today's prices in 2025-2026. What share of the perfect-foresight revenue does it capture?

In [ ]:
# your code here

**Task 4.** A one-hour battery: set E_MAX = 1 and recompute the 2025 perfect-foresight revenue. Is the second hour of storage worth half of the revenue?

In [ ]:
# your code here

## Trading and hedging

In [ ]:
m = pd.read_csv(DATA + 'markets_daily.csv', parse_dates=['date'])
w = m.pivot(index='date', columns='ticker', values='close').loc[:END]
px = w['SP500'].dropna(); vix = w['VIX'].reindex(px.index).ffill()
r = np.log(px).diff().shift(-1).dropna()
ma = (px / px.rolling(200).mean() - 1).reindex(r.index)

**Task 5.** Evaluate the 200-day moving-average rule on 2020 - October 2026 with costs of 0, 5, 20 and 50 basis points per trade and compare its Sharpe ratio with buy-and-hold.

In [ ]:
# your code here

**Task 6.** Delta-hedge the one-month at-the-money call with the 21-day realised volatility instead of the VIX. Compare the standard deviation of the hedging profit and loss with the VIX-based hedge (no transaction costs).

In [ ]:
# your code here